In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
dataset=pd.read_csv('Encoded.csv')

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215890 entries, 0 to 1215889
Data columns (total 96 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Unnamed: 0   1215890 non-null  int64  
 1   Seq          1215890 non-null  int64  
 2   Dur          1215890 non-null  float64
 3   RunTime      1215890 non-null  float64
 4   Mean         1215890 non-null  float64
 5   Sum          1215890 non-null  float64
 6   Min          1215890 non-null  float64
 7   Max          1215890 non-null  float64
 8   sTos         1215676 non-null  float64
 9   dTos         272823 non-null   float64
 10  sTtl         1215676 non-null  float64
 11  dTtl         272823 non-null   float64
 12  sHops        1215676 non-null  float64
 13  dHops        272823 non-null   float64
 14  TotPkts      1215890 non-null  int64  
 15  SrcPkts      1215890 non-null  int64  
 16  DstPkts      1215890 non-null  int64  
 17  TotBytes     1215890 non-null  int64  
 18  Sr

In [20]:
dataset['Label'].value_counts()

Label
Malicious    738153
Benign       477737
Name: count, dtype: int64

In [24]:
drop_cols = [
    "Unnamed: 0", "Seq",
    "Attack Type", "Attack Tool",
    "dTos","dTtl","dHops",
    "SrcGap","DstGap",
    "SrcWin","DstWin",
    "sVid","dVid",
    "SrcTCPBase","DstTCPBase"
]

df = dataset.drop(columns=drop_cols, errors="ignore")

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215890 entries, 0 to 1215889
Data columns (total 81 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   Dur         1215890 non-null  float64
 1   RunTime     1215890 non-null  float64
 2   Mean        1215890 non-null  float64
 3   Sum         1215890 non-null  float64
 4   Min         1215890 non-null  float64
 5   Max         1215890 non-null  float64
 6   sTos        1215676 non-null  float64
 7   sTtl        1215676 non-null  float64
 8   sHops       1215676 non-null  float64
 9   TotPkts     1215890 non-null  int64  
 10  SrcPkts     1215890 non-null  int64  
 11  DstPkts     1215890 non-null  int64  
 12  TotBytes    1215890 non-null  int64  
 13  SrcBytes    1215890 non-null  int64  
 14  DstBytes    1215890 non-null  int64  
 15  Offset      1215890 non-null  int64  
 16  sMeanPktSz  1215890 non-null  float64
 17  dMeanPktSz  1215890 non-null  float64
 18  Load        1215890 no

In [27]:
y = df["Label"]
x = df.drop("Label", axis=1)

In [36]:
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [35]:
df = df.dropna()

In [37]:
from sklearn.model_selection import train_test_split

xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,stratify=y,random_state=42)

In [57]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rf.fit(xtrain, ytrain)

,n_estimators,100
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [40]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_rf = rf.predict(xtest)

print("Accuracy:", accuracy_score(ytest, y_pred_rf))
print(classification_report(ytest, y_pred_rf))

Accuracy: 0.9851466826768869
              precision    recall  f1-score   support

      Benign       1.00      0.96      0.98     95547
   Malicious       0.98      1.00      0.99    147631

    accuracy                           0.99    243178
   macro avg       0.99      0.98      0.98    243178
weighted avg       0.99      0.99      0.99    243178



In [43]:
wrong = ytest != y_pred_rf

In [44]:
wrong_rows = xtest[wrong]

In [45]:
import pandas as pd

errors_df = pd.DataFrame({
    "Actual": ytest[wrong],
    "Predicted": y_pred_rf[wrong]
})

In [46]:
wrong_data = xtest[wrong].copy()
wrong_data["Actual"] = ytest[wrong]
wrong_data["Predicted"] = y_pred_rf[wrong]

In [55]:
print("Total errors:", wrong.sum())
total_correct = (~wrong).sum()
print("Total Correct:", total_correct)

Total errors: 3612
Total Correct: 239566
